#### Generate tf.records from the lsdb file compatible for the model and performs preprocessing.

In [2]:
import os
import numpy as np
import tensorflow as tf 
from lsdb import read_hats
import matplotlib.pyplot as plt
from dask.distributed import Client
from dart.bands.bands import ztf_band
from dart.utils.helper import standardize, generate_data_finetuning
from dart.src.dataset import create_dataset

In [ ]:
#
# Read catalog
#
read_catalog = read_hats('/media3/majumder/dataset/cepheids/hats/zubercal_vcep', )
path_to_store="/media3/majumder/dataset/cepheids/"
#
#
#
catalog_compute = read_catalog._ddf.map_partitions(create_dataset, 
                                                        target=path_to_store,
                                                        bands=ztf_band,
                                                        seed=42,
                                                        min_detec=100,
                                                        train_size=.80,
                                                        max_lcs_per_chunk=100)

with Client() as client:
    catalog_compute.compute(scheduler='processes')


print("\nDone!")

    

#### Generate histogram of the lightcurve for each ZTF-band to determine "min_detec" param in the create_dataset()

In [ ]:
#
#
#
hist = dict()
labels = list()
chunks = list()
filenames = list()
path = f"/media3/majumder/dataset/cepheids/test/"
#
#
#
for p in os.listdir(path):
    for lbl in os.listdir(path+p):
        for cnk in os.listdir(path+p+"/"+lbl):
            filenames.append(path+p+"/"+lbl+'/'+cnk)
# print(filenames)
#
#
#
for k in ztf_band.keys():
    hist[k] = list()
#
dataset = tf.data.TFRecordDataset(filenames)
#
#
#
for rec in dataset:
    #
    example = tf.train.SequenceExample()
    example.ParseFromString(rec.numpy())
    #
    last_index = example.context.feature['last_index'].int64_list.value
    time = example.feature_lists.feature_list['dim_0'].feature[0].float_list.value
    mag = example.feature_lists.feature_list['dim_1'].feature[0].float_list.value
    band_sorted = example.feature_lists.feature_list['dim_3'].feature[0].float_list.value
    last_index = example.context.feature['last_index'].int64_list.value
    id = example.context.feature['id'].int64_list.value[0]
    # 
    j=0
    for i, l in enumerate(hist.keys()):
        #
        temp = last_index[i]+1
        length = temp-j
        hist[l].append(length)
        j = temp
        
print(hist)

In [ ]:
#
#
#
plt.figure(figsize=(8, 10))
plt.suptitle(f"Histogram displaying the lengths of light curves for each ZTF filter")
for i , k in enumerate(hist.keys()):
    # 
    plt.subplot(3, 1, i+1)
    plt.hist(hist[k], bins=30, color="#3492eb", rwidth=0.9)
    plt.xlim(min(hist[k])-100, max(hist[k]))
    plt.title(f"ZTF-{k} filter")
    plt.subplots_adjust(hspace=0.5)
    plt.xlabel("# of detections in each light curve")
# plt.savefig("./cephids/length.png")

#### Determine the ZTF mag_limit and mag_saturation value

In [ ]:
#
#
#
magnitude = list()
labels = list()
chunks = list()
filenames = list()
path = f"/media3/majumder/dataset/multi-class/test/"
#
#
#
for p in os.listdir(path):
    for lbl in os.listdir(path+p):
        for cnk in os.listdir(path+p+"/"+lbl):
            filenames.append(path+p+"/"+lbl+'/'+cnk)

for f in filenames:
    #
    #
    #
    dataset = tf.data.TFRecordDataset(f)
    #
    #
    #
    for rec in dataset:
        #
        example = tf.train.SequenceExample()
        example.ParseFromString(rec.numpy())
        # Convert to TensorFlow tensors
        mag = tf.convert_to_tensor(example.feature_lists.feature_list['dim_1'].feature[0].float_list.value, dtype=tf.float64)
        magerr = tf.convert_to_tensor(example.feature_lists.feature_list['dim_2'].feature[0].float_list.value, dtype=tf.float64)
        #
        new_mag, _ = standardize(mag, magerr)
        magnitude.extend(new_mag.numpy())
            
print(len(magnitude))
# Total available data: 3941899925

In [ ]:
# Calculate 1st and 99th percentiles
percentile_1 = np.percentile(magnitude, 1)
percentile_99 = np.percentile(magnitude, 99)
# Filter data within the 99% range
magnitude_np = np.array(magnitude)
filtered_data = magnitude_np[(magnitude_np >= percentile_1) & (magnitude_np <= percentile_99)]
#
# The min value represents the brightest detection and the max value represents the faintest detection
# mag_saturation = min(filtered_data)
# mag_limit = max(filtered_data)
print(len(filtered_data), min(filtered_data), max(filtered_data))

# The values of the above parameters: 3863061925 -0.6958509378753988 1.5205410620126045

3863061925 -0.6958509378753988 1.5205410620126045


#### Generate data for Finetuning

In [3]:
path_to_read = "/media3/majumder/dataset/cepheids/val/"
path_to_store = "/media3/majumder/dataset/cepheids/finetuning/"
objects_per_chunk = 100
generate_data_finetuning(path_to_read, path_to_store, objects_per_chunk)

2025-04-12 02:56:08.772046: I tensorflow/core/framework/local_rendezvous.cc:405] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence
